# OpenTrace + LangGraph + BBEH (clean)

This notebook is a cleaned/compacted version of the original experiment notebook.

Defaults:
- **Strategy:** `solve_with_PAL_Strategy`
- **Benchmark:** **BBEH** → `bbeh_boolean_expressions` (no GSM8K)
- **Optimization sampling:** **CurriculumBuffer Mode B** (curriculum: current example + last successes)
- **No strategy sweep** and **no end-of-notebook plots/graphs** (optional trace visuals are disabled by default)



In [ ]:
import os

# -----------------------
# Core defaults (edit me)
# -----------------------
BBEH_TASK_NAME = os.getenv("BBEH_TASK_NAME", "bbeh_boolean_expressions")

# Data split (BBEH tasks are stored as JSON "examples"; we just shuffle + slice)
N_TRAIN = int(os.getenv("N_TRAIN", "20"))
N_VAL   = int(os.getenv("N_VAL", "10"))
SEED    = int(os.getenv("SEED", "0"))

# CurriculumBuffer Mode B (curriculum):
# - keep last N successful examples as validation history
# - when optimizing on a failing example, train on (current + history) via accumulation_steps
VALIDATE_ON_LAST_N  = int(os.getenv("VALIDATE_ON_LAST_N", "2"))
ACCUMULATION_STEPS  = int(os.getenv("ACCUMULATION_STEPS", "2"))  # effective_batch_size = 1 + ACCUMULATION_STEPS

# Optimization loop controls
LEARNING_RETRY = int(os.getenv("LEARNING_RETRY", "20"))  # target update-steps per optimize_langgraph() call
MAX_ATTEMPTS   = int(os.getenv("MAX_ATTEMPTS", "10"))    # tries per update-step to get a real parameter change

SKIP_OPTIMIZATION = os.getenv("SKIP_OPTIMIZATION", "0") == "1"

# Output
OUTPUT_FOLDER = os.getenv("OUTPUT_FOLDER", "./trace_runs")

# Optional verbosity toggles (kept OFF by default)
SHOW_MERMAID_GRAPH = os.getenv("SHOW_MERMAID_GRAPH", "0") == "1"
SHOW_OPT_TRACE     = os.getenv("SHOW_OPT_TRACE", "0") == "1"  # Trace backward visuals

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Config:")
print(f"  {BBEH_TASK_NAME=}")
print(f"  {N_TRAIN=}, {N_VAL=}, {SEED=}")
print(f"  {VALIDATE_ON_LAST_N=}, {ACCUMULATION_STEPS=}")
print(f"  {LEARNING_RETRY=}, {MAX_ATTEMPTS=}")
print(f"  {SKIP_OPTIMIZATION=}")
print(f"  {OUTPUT_FOLDER=}")

Config:
  BBEH_TASK_NAME='bbeh_boolean_expressions'
  N_TRAIN=20, N_VAL=10, SEED=0
  VALIDATE_ON_LAST_N=2, ACCUMULATION_STEPS=2
  LEARNING_RETRY=20, MAX_ATTEMPTS=10
  SKIP_OPTIMIZATION=False
  OUTPUT_FOLDER='./trace_runs'


In [ ]:
import os, sys

# -----------------------
# Optional: install deps
# -----------------------
# If you are in a fresh Colab/runtime, you likely need:
#
import sys
if IN_COLAB:
  # test if setup has already been done : reset by !rm -rf /content/Trace
  if not os.path.exists('/content/Trace'):
    print("Setting up Trace...")
    %pip install langgraph langchain langchain_openai datasets tqdm langchain_community litellm dspy black
    %alias git git
    %alias sed sed
    %git clone https://github.com/AgentOpt/OpenTrace.git Trace
    %cd Trace
    %git pull origin experimental && git checkout experimental
    %sed -i 's/python_requires=">=3.13"/python_requires=">=3.12"/' setup.py
    %pip install -e .
  sys.path.append('/content/Trace')
else:
    sys.path.append(os.path.expanduser("~/trace/Trace"))
#
# Also clone BBEH tasks:
!git clone https://github.com/google-deepmind/bbeh.git

# Try to auto-add a local Trace repo path (edit if needed)
IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

trace_repo = os.getenv("TRACE_REPO", "/content/Trace" if IN_COLAB else os.path.expanduser("~/trace/Trace"))
if os.path.exists(trace_repo) and trace_repo not in sys.path:
    sys.path.append(trace_repo)

# Soft-import display (avoid hard dependency on IPython)
try:
    from IPython.display import display  # type: ignore
except Exception:
    def display(*args, **kwargs):  # noqa: D401
        return None

print(f"{IN_COLAB=}, trace_repo_exists={os.path.exists(trace_repo)}")

Setting up Trace...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.4/312.4 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

# -----------------------
# LLM config (defaults)
# -----------------------
LLM_SERVICE = os.getenv("LLM_SERVICE", "openrouter")   # "openai" | "openrouter" | "customllm"
LLM_GENERAL_MODEL = os.getenv("LLM_GENERAL_MODEL", "openai/gpt-5-nano")

# API keys: prefer env vars (Colab users can also use google.colab.userdata)
def _get_secret(name: str) -> str | None:
    try:
        from google.colab import userdata  # type: ignore
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.getenv(name)

OPENAI_API_KEY     = _get_secret("OPENAI_API_KEY")
OPENROUTER_API_KEY = _get_secret("OPENROUTER_API_KEY")
CUSTOMLLM_API_KEY  = _get_secret("CUSTOMLLM_API_KEY")
CUSTOMLLM_URL      = os.getenv("CUSTOMLLM_URL", "http://localhost:4000/")  # if you use a local proxy

if LLM_SERVICE == "openrouter":
    if not OPENROUTER_API_KEY:
        raise ValueError("OPENROUTER_API_KEY missing (set env var or Colab secret).")
    os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
    os.environ["OPENAI_API_KEY"]  = OPENROUTER_API_KEY
elif LLM_SERVICE == "customllm":
    if not CUSTOMLLM_API_KEY:
        raise ValueError("CUSTOMLLM_API_KEY missing (set env var or Colab secret).")
    os.environ["OPENAI_BASE_URL"] = CUSTOMLLM_URL
    os.environ["OPENAI_API_KEY"]  = CUSTOMLLM_API_KEY
else:
    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY missing (set env var or Colab secret).")
    os.environ["OPENAI_BASE_URL"] = "https://api.openai.com/v1"
    os.environ["OPENAI_API_KEY"]  = OPENAI_API_KEY

llm = ChatOpenAI(model_name=LLM_GENERAL_MODEL, temperature=0)

def llm_call(prompt: str, system_instructions: str = "") -> str:
    msgs = [HumanMessage(content=prompt)]
    if system_instructions:
        msgs.insert(0, SystemMessage(content=system_instructions))
    return llm.invoke(msgs).content

print("LLM ready:", {"service": LLM_SERVICE, "model": LLM_GENERAL_MODEL})

LLM ready: {'service': 'openrouter', 'model': 'openai/gpt-5-nano'}


In [ ]:
import os, json, random, inspect
from copy import deepcopy

# ---- Trace imports (OpenTrace / opto) ----
try:
    from opto.trace import node, bundle
    from opto.trace.bundle import FunModule
    from opto.optimizers.optoprime_v2 import OptoPrimeV2 as OptoPrime
    from opto.trainer.guide import Guide as _TraceGuide
    from opto.trainer.algorithms.basic_algorithms import Minibatch as _TraceMinibatch
except Exception as e:
    raise ImportError(
        "Could not import OpenTrace (opto.*). "
        "Make sure OpenTrace is installed and TRACE_REPO is on sys.path."
    ) from e


# -----------------------
# Small helpers
# -----------------------
def set_dict(state: dict, key, value):
    (state.data if hasattr(state, "data") else state)[key] = value

def get_no_node(x):
    return x.data if hasattr(x, "data") else x

def _snapshot_params(parameters):
    snap = {}
    for p in parameters:
        try:
            snap[p.name] = deepcopy(p.data)
        except Exception:
            snap[p.name] = p.data
    return snap

def _params_changed(before, after) -> bool:
    if before.keys() != after.keys():
        return True
    for k in before.keys():
        if str(before[k]) != str(after[k]):
            return True
    return False

def _replace_in_scope_by_identity(scope: dict, old_obj, new_obj) -> list[str]:
    replaced = []
    for k, v in list(scope.items()):
        if v is old_obj:
            scope[k] = new_obj
            replaced.append(k)
    return replaced

def bind_function(func, *, trainable=True, traceable_code=True, allow_external_dependencies=True):
    """Safely bundle() a python function into a Trace FunModule (only once)."""
    if func is None or not callable(func):
        return func
    if isinstance(func, FunModule):
        return func
    fm = bundle(trainable=trainable,
                traceable_code=traceable_code,
                allow_external_dependencies=allow_external_dependencies)(func)
    # Preserve signature for nicer debugging
    try:
        fm.__signature__ = inspect.signature(fm._fun)
    except Exception:
        pass
    return fm


# -----------------------
# Guide: graph output -> (score, feedback)
# -----------------------
class LangGraphGuide(_TraceGuide):
    def __init__(self, feedback_func, *, answer_key="final_answer", allowed_answer_set=None):
        self.feedback_func = feedback_func
        self.answer_key = answer_key
        self.allowed = allowed_answer_set

    def get_feedback(self, query, response, reference, **kwargs):
        # response is usually a dict: {"final_answer": <node>}
        try:
            if isinstance(response, dict) or (hasattr(response, "data") and isinstance(get_no_node(response), dict)):
                extracted = get_no_node(get_no_node(response)[self.answer_key])
            else:
                extracted = get_no_node(response)
        except Exception:
            extracted = get_no_node(response)

        if self.allowed is not None:
            ok, fb = self.feedback_func(extracted, reference, self.allowed)
        else:
            ok, fb = self.feedback_func(extracted, reference)
        return float(bool(ok)), fb

    def copy(self):
        return LangGraphGuide(self.feedback_func, answer_key=self.answer_key, allowed_answer_set=self.allowed)


# -----------------------
# CurriculumBuffer
# -----------------------
class CurriculumBuffer:
    """Mode A (fixed pool) if training_pool is provided; Mode B (curriculum) otherwise."""
    def __init__(self, training_pool=None, *, history_size=2, sample_with_replacement=True, seed=None):
        self.pool = list(training_pool) if training_pool else []
        self.history = []
        self.history_size = int(history_size)
        self.replacement = bool(sample_with_replacement)
        self._rng = random.Random(seed)

    @property
    def is_fixed_pool(self) -> bool:
        return len(self.pool) > 0

    def add_success(self, example: dict):
        self.history.append(example)
        if len(self.history) > self.history_size:
            self.history.pop(0)

    def sample_batch(self, batch_size: int, *, current_question=None, current_solution=None) -> list[dict]:
        if self.is_fixed_pool:
            k = batch_size if self.replacement else min(batch_size, len(self.pool))
            return self._rng.choices(self.pool, k=k) if self.replacement else self._rng.sample(self.pool, k=k)

        # Mode B: current + recent successes
        batch = []
        max_steps = min(batch_size, 1 + len(self.history))
        for i in range(max_steps):
            if i == 0:
                batch.append({"question": current_question, "solution": current_solution})
            else:
                ex = self.history[-i]
                batch.append({"question": ex["question"], "solution": ex.get("solution", ex.get("answer"))})
        return batch


# -----------------------
# Trainer
# -----------------------
class LangGraphTrainer(_TraceMinibatch):
    def __init__(self, *, graph_root_function: str, graph_agents_functions: list[str], scope: dict,
                 optimizer, parameters: list):
        object.__init__(self)
        self.root_name = graph_root_function
        self.agent_names = list(graph_agents_functions)
        self.scope = scope
        self.optimizer = optimizer
        self.parameters = list(parameters)

        # originals for corruption guard / rollback
        self._original_root = scope[graph_root_function]
        self._original_agents = {n: scope[n] for n in graph_agents_functions if n in scope}

    def restore_originals(self):
        self.scope[self.root_name] = self._original_root
        for name, orig in self._original_agents.items():
            self.scope[name] = orig

    def _check_corruption(self) -> bool:
        restored = False
        for name in self.agent_names:
            agent = self.scope.get(name)
            if isinstance(agent, FunModule) and getattr(agent, "_fun", None) is None:
                print(f"⚠️ corruption: '{name}' has ._fun=None. Restoring original.")
                self.scope[name] = self._original_agents[name]
                restored = True
        return restored

    def _run_one(self, question, solution, guide: LangGraphGuide):
        answer_key = guide.answer_key
        try:
            answer = self.scope[self.root_name](question)
            score, feedback = guide.get_feedback(question, answer, solution)
            ok = score >= 1.0
        except Exception as e:
            ok = False
            feedback = f"ERROR: {e}"
            answer = {answer_key: node("DUMMY_ANSWER")}
        return answer, ok, feedback

    def train(self, *, guide: LangGraphGuide, buffer: CurriculumBuffer,
              question=None, solution=None,
              target_updates=20, max_attempts=10, batch_size=3,
              test_optimization=True, stop_on_success=True,
              run_dir=".", save_steps=True,
              validation_set=None):
        if validation_set is None:
            validation_set = []

        answer_key = guide.answer_key
        best_state = None
        last_state = None
        history = []
        modified = False
        updates_done = 0
        global_attempt = 0

        os.makedirs(run_dir, exist_ok=True)

        while updates_done < int(target_updates):
            step_attempt = 0
            step_changed = False

            while step_attempt < int(max_attempts) and not step_changed:
                step_attempt += 1
                global_attempt += 1
                attempt = global_attempt
                print(f"[opt] attempt={attempt} update_step={updates_done+1}/{target_updates} try={step_attempt}/{max_attempts}")

                self.optimizer.zero_feedback()

                # minibatch
                batch_examples = buffer.sample_batch(
                    int(batch_size),
                    current_question=question,
                    current_solution=solution,
                )

                answers = []
                feedbacks = []
                batch_all_correct = True

                for ex in batch_examples:
                    eq = ex["question"]
                    es = ex.get("solution", ex.get("answer"))
                    ans, ok, fb = self._run_one(eq, es, guide)
                    batch_all_correct = batch_all_correct and ok
                    answers.append(ans)
                    feedbacks.append(fb)

                # aggregate feedback
                if len(feedbacks) == 1:
                    common_feedback = feedbacks[0]
                else:
                    common_feedback = "\n".join([f"Feedback #{i+1}: {fb}" for i, fb in enumerate(feedbacks)])

                # backward
                for ans in answers:
                    ans_node = ans.get(answer_key, ans) if isinstance(ans, dict) else ans
                    if not hasattr(ans_node, "backward"):
                        ans_node = node(str(ans_node))
                    self.optimizer.backward(
                        ans_node,
                        common_feedback,
                        visualize=bool(SHOW_OPT_TRACE),
                        print_limit=30,
                    )

                # step + change detection
                before = _snapshot_params(self.parameters)
                self.optimizer.step(verbose=True)
                after = _snapshot_params(self.parameters)
                step_changed = _params_changed(before, after)

                # corruption guard
                if self._check_corruption():
                    step_changed = False

                if not step_changed:
                    print("[opt] no parameter change, retrying...")
                    continue

                # record successful update
                updates_done += 1
                modified = True
                last_state = {p.name: p.data for p in self.parameters}

                # compute val acc (optional)
                val_acc = None
                if validation_set:
                    n_ok = 0
                    for v in validation_set:
                        _, vok, _ = self._run_one(v["question"], v.get("solution", v.get("answer")), guide)
                        n_ok += int(vok)
                    val_acc = n_ok / float(len(validation_set))

                # save step snapshot (optional)
                if save_steps:
                    try:
                        step_path = os.path.join(run_dir, f"step_{updates_done:03d}_state.txt")
                        with open(step_path, "w") as f:
                            for nm, val in last_state.items():
                                f.write(f"{nm}: {val}\n")
                    except Exception as e:
                        print(f"⚠️ could not save step state: {e}")

                # test_optimization gate: current example + validation_set must pass
                if test_optimization and question is not None:
                    _, cur_ok, cur_fb = self._run_one(question, solution, guide)
                    val_ok = True
                    for v in validation_set:
                        _, vok, _ = self._run_one(v["question"], v.get("solution", v.get("answer")), guide)
                        if not vok:
                            val_ok = False
                            break
                    if cur_ok and val_ok:
                        best_state = last_state
                        print("[opt] gate PASS:", cur_fb)
                        if stop_on_success:
                            # write history entry before stopping
                            hist_entry = {
                                "update_step": updates_done,
                                "attempt": attempt,
                                "batch_size": int(batch_size),
                                "mode": "fixed" if buffer.is_fixed_pool else "curriculum",
                                "train_batch_all_correct": batch_all_correct,
                                "val_acc": val_acc,
                                "gate_pass": True,
                            }
                            history.append(hist_entry)
                            with open(os.path.join(run_dir, "history.jsonl"), "a") as f:
                                f.write(json.dumps(hist_entry, default=str) + "\n")
                            return modified, history, best_state, last_state

                # history entry (normal)
                hist_entry = {
                    "update_step": updates_done,
                    "attempt": attempt,
                    "batch_size": int(batch_size),
                    "mode": "fixed" if buffer.is_fixed_pool else "curriculum",
                    "train_batch_all_correct": batch_all_correct,
                    "val_acc": val_acc,
                    "gate_pass": bool(best_state is not None),
                }
                history.append(hist_entry)
                try:
                    with open(os.path.join(run_dir, "history.jsonl"), "a") as f:
                        f.write(json.dumps(hist_entry, default=str) + "\n")
                except Exception:
                    pass

                if stop_on_success and best_state is not None:
                    return modified, history, best_state, last_state

            if not step_changed:
                print(f"⚠️ stopping early: couldn't get a parameter update after {max_attempts} tries.")
                break

        return modified, history, best_state, last_state


# -----------------------
# optimize_langgraph (thin facade)
# -----------------------
def optimize_langgraph(
    *,
    graph_root_function: str,
    graph_agents_functions: list[str],
    question: str,
    solution: str,
    graph_prompts_list=None,
    answer_feedback_func=None,
    allowed_answer_set=None,
    answer_key="final_answer",
    validation_set=None,
    # Mode A vs B
    training_pool=None,
    batch_size=None,
    accumulation_steps=1,
    sample_with_replacement=True,
    seed=None,
    # Loop controls
    updating_steps=None,
    retry=5,
    max_attempts=10,
    stop_on_success=True,
    test_optimization=True,
    train_graph_agents_functions=True,
    memory_size=1,
    save_steps=True,
    dump_prefix="",
    output_folder=None,
    scope=None,
    optimizer_cls=None,
    trainer_cls=None,
):
    if optimizer_cls is None:
        optimizer_cls = OptoPrime
    if trainer_cls is None:
        trainer_cls = LangGraphTrainer
    if scope is None:
        scope = globals()
    if validation_set is None:
        validation_set = []
    if seed is not None:
        random.seed(seed)

    # Bind agents + prompts
    if isinstance(scope.get(graph_root_function), FunModule):
        scope[graph_root_function] = scope[graph_root_function]._fun

    parameters = []
    for name in graph_agents_functions:
        if name not in scope:
            raise KeyError(f"'{name}' not found in scope.")
        scope[name] = bind_function(scope[name], trainable=train_graph_agents_functions)
        parameters.extend(scope[name].parameters())

    if graph_prompts_list is not None:
        for i, prompt in enumerate(list(graph_prompts_list)):
            if hasattr(prompt, "data") and hasattr(prompt, "name"):
                parameters.append(prompt)
                continue
            new_prompt = node(str(prompt), trainable=True)
            _replace_in_scope_by_identity(scope, prompt, new_prompt)
            graph_prompts_list[i] = new_prompt
            parameters.append(new_prompt)

    if not parameters:
        raise ValueError("No trainable parameters found (agents/prompts list is empty).")

    # Optimizer, guide, buffer
    opt = optimizer_cls(
        parameters,
        memory_size=memory_size,
        objective=[
            "Improve the agent so it solves the task reliably.",
            "Prefer simple, robust edits to prompts/code."
        ],
    )

    guide = LangGraphGuide(
        feedback_func=answer_feedback_func,
        answer_key=answer_key,
        allowed_answer_set=allowed_answer_set,
    )

    effective_batch_size = int(batch_size) if batch_size is not None else max(1, 1 + int(accumulation_steps))

    buffer = CurriculumBuffer(
        training_pool=training_pool,
        history_size=max(len(validation_set), 2) if validation_set else 2,
        sample_with_replacement=sample_with_replacement,
        seed=seed,
    )
    # Pre-seed curriculum history from validation_set (Mode B)
    if (not buffer.is_fixed_pool) and validation_set:
        for v in validation_set:
            buffer.add_success(v)

    target_updates = int(updating_steps) if updating_steps is not None else int(retry)
    _max_attempts = int(max_attempts)

    # Run directory
    base_dir = output_folder or "."
    os.makedirs(base_dir, exist_ok=True)
    run_name = (
        f"{dump_prefix}{graph_root_function}"
        f"__mode-{'fixed' if buffer.is_fixed_pool else 'curr'}"
        f"__bs{effective_batch_size}"
        f"__updates{target_updates}"
        f"__maxA{_max_attempts}"
        f"__mem{memory_size}"
        f"__seed{seed if seed is not None else 'none'}"
    )
    run_dir = os.path.join(base_dir, run_name)
    os.makedirs(run_dir, exist_ok=True)

    # Train
    trainer = trainer_cls(
        graph_root_function=graph_root_function,
        graph_agents_functions=graph_agents_functions,
        scope=scope,
        optimizer=opt,
        parameters=parameters,
    )
    modified, history, best_state, last_state = trainer.train(
        guide=guide,
        buffer=buffer,
        question=question,
        solution=solution,
        target_updates=target_updates,
        max_attempts=_max_attempts,
        batch_size=effective_batch_size,
        test_optimization=test_optimization,
        stop_on_success=stop_on_success,
        save_steps=save_steps,
        run_dir=run_dir,
        validation_set=validation_set,
    )

    chosen_state = best_state if best_state is not None else last_state
    dump_filename = None
    if modified and chosen_state is not None:
        dump_filename = os.path.join(run_dir, "best_state.txt")
        with open(dump_filename, "w") as f:
            for nm, val in chosen_state.items():
                f.write(f"{nm}: {val}\n")

    # Rollback if we didn't get a passing best_state (keeps semantics stable)
    if (not test_optimization) or (best_state is None):
        trainer.restore_originals()

    return modified, dump_filename, history, chosen_state, run_dir

In [ ]:
import re
from langgraph.graph import StateGraph, START, END

# -----------------------
# Strategy: PAL
# -----------------------
prompt_parse_problem = node(
    "Read the problem and write Python code that sets a variable named `result` to the final answer.\n"
    "- Output ONLY valid Python (no markdown fences).\n"
    "- If the task is multiple-choice, set result to the option label exactly (e.g., '(A)').\n\n"
    "Problem:\n",
    trainable=True,
    description="PAL prompt that generates python code producing a `result`."
)

def parse_problem(state: dict):
    question = get_no_node(state.get("question", ""))
    prompt = prompt_parse_problem + question
    code_str = llm_call(get_no_node(prompt))
    return {"code": code_str.strip(), "question": question}

def execute_code(state: dict):
    def strip_python_tags(code: str) -> str:
        return re.sub(
            r'(?s)(?:.*?```(?:python)?\s*\n(.*?)(?:\n```.*)?|(.*))\Z',
            lambda m: m.group(1) if m.group(1) is not None else m.group(2),
            code,
        )

    update = {}
    try:
        code_to_run = strip_python_tags(get_no_node(state.get("code", "")))
        local_vars = {}
        exec(code_to_run, {}, local_vars)
        local_vars.pop("__builtins__", None)

        if "result" in local_vars:
            update["final_answer"] = node(local_vars["result"])
        elif len(local_vars) == 1:
            update["final_answer"] = node(next(iter(local_vars.values())))
        else:
            update["final_answer"] = node(None)

    except Exception as e:
        update["final_answer"] = node(None)
        update["error"] = str(e)

    return update

def create_graph_solve_with_PAL_Strategy():
    g = StateGraph(dict)
    g.add_node("parse", parse_problem)
    g.add_node("calculate", execute_code)
    g.add_edge(START, "parse")
    g.add_edge("parse", "calculate")
    g.add_edge("calculate", END)
    return g

def solve_with_PAL_Strategy(problem: str) -> dict:
    g = create_graph_solve_with_PAL_Strategy()
    compiled = g.compile()

    # NOTE: graph visualization disabled by default
    if SHOW_MERMAID_GRAPH:
        try:
            from IPython.display import Image, display  # type: ignore
            display(Image(compiled.get_graph(xray=1).draw_mermaid_png()))
        except Exception:
            pass

    result = compiled.invoke({"question": get_no_node(problem)})
    if "final_answer" not in result:
        return {"final_answer": node("No solution found")}
    if isinstance(result["final_answer"], str):
        return {"final_answer": node(result["final_answer"])}
    return result

# Default "graph spec" for optimize_langgraph
GRAPH_ROOT = "solve_with_PAL_Strategy"
GRAPH_AGENTS = ["parse_problem", "execute_code"]
GRAPH_PROMPTS = [prompt_parse_problem]

In [ ]:
import os, json, random, string

# -----------------------
# BBEH dataset loader
# -----------------------
# Repo layout varies slightly depending on how you clone / where you run.
def _find_bbeh_tasks_dir() -> str:
    candidates = [
        "bbeh/benchmark_tasks",
        "bbeh/bbeh/benchmark_tasks",
        "benchmark_tasks",
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(
        "Could not locate BBEH benchmark_tasks folder.\n"
        "Clone the repo first, e.g. `git clone https://github.com/google-deepmind/bbeh.git`."
    )

bbeh_tasks_dir = _find_bbeh_tasks_dir()
print("BBEH tasks dir:", bbeh_tasks_dir)

# For this notebook we only need the task(s) with constrained outputs.
LIMITED_BBEH_OUTPUT_TASKS = {
    "bbeh_boolean_expressions": {"(A)", "(B)", "(C)", "(D)", "(E)"},
}

def normalize_answer(ans) -> str:
    if ans is None:
        return ""
    ans = str(ans).strip().lower()
    ans = ans.translate(str.maketrans("", "", string.punctuation))
    ans = ans.replace(" ", "")
    return ans

def feedback_answer_bbeh(predicted, target, allowed_set=None):
    pred_raw = get_no_node(predicted)
    pred_norm = normalize_answer(pred_raw)
    target_norm = normalize_answer(target)

    allowed_norm = None
    if allowed_set:
        allowed_norm = {normalize_answer(a) for a in allowed_set}

    if pred_norm == target_norm:
        return True, f"SUCCESS: '{pred_raw}'"
    msg = f"FAILED: '{pred_raw}' != '{target}'. Fix the code/prompt to solve similar problems."
    if allowed_norm is not None and pred_norm not in allowed_norm:
        msg += f" (final answer must be one of: {sorted(allowed_set)})"
    return False, msg

def load_bbeh_examples(task_name: str, *, n_train: int, n_val: int, seed: int = 0):
    task_path = os.path.join(bbeh_tasks_dir, task_name, "task.json")
    if not os.path.exists(task_path):
        raise FileNotFoundError(f"Task not found: {task_path}")

    with open(task_path, "r") as f:
        task = json.load(f)

    examples = task.get("examples", [])
    rng = random.Random(seed)
    rng.shuffle(examples)

    allowed = LIMITED_BBEH_OUTPUT_TASKS.get(task_name)
    def _format_q(q: str) -> str:
        if allowed:
            return q + f"\n\nAllowed final answer: {sorted(allowed)}"
        return q

    # Convert to the {question, solution} format used by optimize_langgraph
    items = [{"question": _format_q(ex["input"]), "solution": ex["target"]} for ex in examples]

    train = items[:n_train]
    val   = items[n_train:n_train + n_val]
    return train, val, allowed

train_set, val_set, allowed_set = load_bbeh_examples(
    BBEH_TASK_NAME,
    n_train=N_TRAIN,
    n_val=N_VAL,
    seed=SEED,
)

print(f"Loaded {len(train_set)} train and {len(val_set)} val examples for {BBEH_TASK_NAME}")

BBEH tasks dir: bbeh/bbeh/benchmark_tasks
Loaded 20 train and 10 val examples for bbeh_boolean_expressions


In [ ]:
from typing import List, Dict, Tuple

def run_solver_on_example(ex: dict) -> Tuple[bool, str, str]:
    out = solve_with_PAL_Strategy(ex["question"])
    pred = get_no_node(out.get("final_answer"))
    ok, fb = feedback_answer_bbeh(pred, ex["solution"], allowed_set)
    return ok, str(pred), fb

def evaluate(examples: List[dict], *, name: str) -> float:
    n_ok = 0
    for i, ex in enumerate(examples, 1):
        ok, pred, fb = run_solver_on_example(ex)
        n_ok += int(ok)
        print(f"[{name}] {i:02d}/{len(examples)} ok={ok} pred={pred} :: {fb}")
    acc = n_ok / max(1, len(examples))
    print(f"[{name}] accuracy = {acc:.3f} ({n_ok}/{len(examples)})")
    return acc

# -----------------------
# Baseline
# -----------------------
baseline_acc = evaluate(val_set, name="baseline/val")

# -----------------------
# Curriculum training (Mode B)
# -----------------------
if SKIP_OPTIMIZATION:
    print("SKIP_OPTIMIZATION=1 -> skipping optimization/training.")
else:
    last_successes: List[dict] = []

    for idx, ex in enumerate(train_set, 1):
        ok, pred, fb = run_solver_on_example(ex)
        print(f"[train] {idx:02d}/{len(train_set)} ok={ok} pred={pred} :: {fb}")

        if ok:
            last_successes.append(ex)
            last_successes = last_successes[-VALIDATE_ON_LAST_N:]
            continue

        # Optimize on the failing example, with validation on last successes (curriculum)
        modified, dump_file, history, chosen_state, run_dir = optimize_langgraph(
            graph_root_function=GRAPH_ROOT,
            graph_agents_functions=GRAPH_AGENTS,
            graph_prompts_list=GRAPH_PROMPTS,
            question=ex["question"],
            solution=ex["solution"],
            answer_feedback_func=feedback_answer_bbeh,
            allowed_answer_set=allowed_set,
            validation_set=last_successes,
            accumulation_steps=ACCUMULATION_STEPS,
            retry=LEARNING_RETRY,
            max_attempts=MAX_ATTEMPTS,
            test_optimization=True,
            stop_on_success=True,
            seed=SEED,
            dump_prefix=f"BBEH_{BBEH_TASK_NAME}__PAL__",
            output_folder=OUTPUT_FOLDER,
        )

        print("[train] optimize_langgraph:", {"modified": modified, "dump_file": dump_file, "run_dir": run_dir})
        if history:
            print("[train] last history entry:", history[-1])

        # Re-test the current example after optimization
        ok2, pred2, fb2 = run_solver_on_example(ex)
        print(f"[train] after-opt ok={ok2} pred={pred2} :: {fb2}")

        if ok2:
            last_successes.append(ex)
            last_successes = last_successes[-VALIDATE_ON_LAST_N:]

# -----------------------
# Post-training eval
# -----------------------
final_acc = evaluate(val_set, name="final/val")

print("Summary:", {"baseline_val_acc": baseline_acc, "final_val_acc": final_acc})

[baseline/val] 01/10 ok=False pred=None :: FAILED: 'None' != '(A)'. Fix the code/prompt to solve similar problems. (final answer must be one of: ['(A)', '(B)', '(C)', '(D)', '(E)'])
[baseline/val] 02/10 ok=True pred=(E) :: SUCCESS: '(E)'
[baseline/val] 03/10 ok=True pred=(C) :: SUCCESS: '(C)'
[baseline/val] 04/10 ok=False pred=(C) :: FAILED: '(C)' != '(D)'. Fix the code/prompt to solve similar problems.
[baseline/val] 05/10 ok=True pred=(B) :: SUCCESS: '(B)'
[baseline/val] 06/10 ok=False pred=(D) :: FAILED: '(D)' != '(E)'. Fix the code/prompt to solve similar problems.
[baseline/val] 07/10 ok=False pred=None :: FAILED: 'None' != '(E)'. Fix the code/prompt to solve similar problems. (final answer must be one of: ['(A)', '(B)', '(C)', '(D)', '(E)'])
[baseline/val] 08/10 ok=False pred=(E) :: FAILED: '(E)' != '(C)'. Fix the code/prompt to solve similar problems.
[baseline/val] 09/10 ok=False pred=(E) :: FAILED: '(E)' != '(B)'. Fix the code/prompt to solve similar problems.
